# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imatiq/ML_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.



```
# This is formatted as code
```

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** A page is worth reviewing first if it is still pulling real
search visibility (so a fix actually matters), and at least one of the following is true: it
hasn't been touched in a long time, it's thin for how much it's seen, it's already ranking
well enough that a push could move it further, or it's ranking decently but people aren't
clicking it. More qualifying conditions and more visibility push it further up the queue.

I deliberately keep `trend_direction` / `trend_pct` out of the rule and the reason codes —
those are the columns the label (`is_declining_label`) is built from, so using them here would
just be reading the answer key.

**Reason codes it can output** (a scored page can carry more than one):

| Reason code | Fires when |
|---|---|
| `stale_but_visible` | `days_since_last_update >= 180` and `impressions_90d >= 500` |
| `thin_but_visible` | `0 < word_count < 1200` and `impressions_90d >= 500` |
| `rankable_position` | `0 < avg_position <= 20` and `impressions_90d >= 500` |
| `ranking_low_ctr` | `0 < avg_position <= 20`, `ctr < 0.5` and `impressions_90d >= 500` |
| `low_priority` | none of the above fired (not visible enough, or no gap found) |


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path("data/raw/content_refresh_anonymized.csv")

# Ensure the directory exists
RAW_PATH.parent.mkdir(parents=True, exist_ok=True)

# Check if the file exists before trying to read it
if RAW_PATH.exists():
    df = pd.read_csv(RAW_PATH)

    # Sanity check for section 1: the columns my rule reads from are all present,
    # and the label-source columns are noted but NOT going into the rule.
    rule_inputs = ["impressions_90d", "days_since_last_update", "word_count", "avg_position", "ctr"]
    label_source_cols = ["trend_direction", "trend_pct"]  # off-limits as features -- see rule note above

    print("rows, cols:", df.shape)
    print("rule inputs present:", all(c in df.columns for c in rule_inputs))
    print("label-source columns excluded from rule:", label_source_cols)
    print(df[rule_inputs].describe().T[["count", "min", "50%", "max"]])
else:
    print(f"Error: The file '{RAW_PATH}' was not found.")
    print("Please upload 'content_refresh_anonymized.csv' to the 'data/raw/' directory in your Colab environment and run this cell again.")

rows, cols: (34668, 44)
rule inputs present: True
label-source columns excluded from rule: ['trend_direction', 'trend_pct']
                          count  min      50%     max
days_since_last_update  34667.0  1.0    20.00   373.0
word_count              25800.0  1.3  2878.00  9546.0
avg_position            34667.0  0.0    10.80   245.0
ctr                     34667.0  0.0     0.07   100.0


/tmp/ipykernel_4121/4188094201.py:12: DtypeWarning: Columns (5,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW_PATH)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the label ONLY for evaluation (never a feature) -- exactly how the pipeline defines it
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Convert 'impressions_90d' to numeric, coercing errors to NaN, then fill NaN with 0
df["impressions_90d"] = pd.to_numeric(df["impressions_90d"], errors='coerce').fillna(0)

visible  = (df["impressions_90d"] >= 500).astype(int)
stale    = (df["days_since_last_update"] >= 180).astype(int)
thin     = ((df["word_count"] > 0) & (df["word_count"] < 1200)).astype(int)
rankable = ((df["avg_position"] > 0) & (df["avg_position"] <= 20)).astype(int)
low_ctr  = ((df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)).astype(int)

# Transparent score: no fitted weights. Visibility gates everything (0 if not visible),
# each qualifying condition adds to the multiplier, then scaled by log(impressions) so that,
# among equally-qualified pages, the ones with more real traffic sort first.
df["action_score"] = visible * (2 * stale + thin + 2 * rankable + low_ctr) * np.log1p(df["impressions_90d"])

def reason_codes(row) -> str:
    codes = []
    if row["stale"] and row["visible"]:
        codes.append("stale_but_visible")
    if row["thin"] and row["visible"]:
        codes.append("thin_but_visible")
    if row["rankable"] and row["visible"]:
        codes.append("rankable_position")
    if row["low_ctr"] and row["visible"]:
        codes.append("ranking_low_ctr")
    if not codes:
        codes.append("low_priority")
    return "|".join(codes)

def suggested_action(codes: str) -> str:
    parts = set(codes.split("|"))
    if "thin_but_visible" in parts:
        return "expand_and_refresh"
    if "ranking_low_ctr" in parts:
        return "refresh_title_and_meta"
    if "rankable_position" in parts or "stale_but_visible" in parts:
        return "refresh"
    return "monitor"

df["visible"], df["stale"], df["thin"], df["rankable"], df["low_ctr"] = visible, stale, thin, rankable, low_ctr
df["reason_codes"] = df.apply(reason_codes, axis=1)
df["suggested_action"] = df["reason_codes"].apply(suggested_action)
df["action_rank"] = df["action_score"].rank(method="first", ascending=False).astype(int)

out = df.sort_values("action_rank").reset_index(drop=True)

output_columns = [
    "content_id", "client_id", "action_rank", "action_score",
    "reason_codes", "suggested_action", "is_declining_label",
    "impressions_90d", "avg_position", "ctr",
    "days_since_last_update", "word_count", "content_type",
]
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
out[output_columns].to_csv(output_path, index=False)

def precision_at_k(sorted_labels, k):
    return sorted_labels.head(k).mean()

base_rate = df["is_declining_label"].mean()
p20 = precision_at_k(out["is_declining_label"], 20)
p50 = precision_at_k(out["is_declining_label"], 50)

print(f"Wrote {output_path} -- {len(out)} rows")
print(f"Base rate (share declining, whole dataset): {base_rate:.3f}")
print(f"Precision@20: {p20:.3f}")
print(f"Precision@50: {p50:.3f}")
print()
print(out["reason_codes"].value_counts())

Wrote work/outputs/baseline_action_score.csv -- 34668 rows
Base rate (share declining, whole dataset): 0.541
Precision@20: 0.650
Precision@50: 0.500

reason_codes
low_priority                                           20715
rankable_position|ranking_low_ctr                      11279
rankable_position                                       2602
thin_but_visible|rankable_position|ranking_low_ctr        20
thin_but_visible                                          16
thin_but_visible|rankable_position                        16
stale_but_visible|rankable_position|ranking_low_ctr       12
stale_but_visible                                          8
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = out.head(20)[[
    "action_rank", "content_id", "reason_codes", "suggested_action",
    "is_declining_label", "impressions_90d", "avg_position", "ctr",
    "days_since_last_update", "word_count",
]]

wrong_picks = top20[top20["is_declining_label"] == 0]

print(f"Top-20 precision: {top20['is_declining_label'].mean():.2f}  ({top20['is_declining_label'].sum()}/20 correct)")
print(f"Wrong picks in top 20: {len(wrong_picks)}")
print(f"Every top-20 row carries 'rankable_position': "
      f"{top20['reason_codes'].str.contains('rankable_position').all()}")
print(f"Every top-20 row carries 'ranking_low_ctr': "
      f"{top20['reason_codes'].str.contains('ranking_low_ctr').all()}")
print(f"'stale_but_visible' or 'thin_but_visible' anywhere in top 20: "
      f"{top20['reason_codes'].str.contains('stale_but_visible|thin_but_visible').any()}")
print()
top20

Top-20 precision: 0.65  (13/20 correct)
Wrong picks in top 20: 7
Every top-20 row carries 'rankable_position': True
Every top-20 row carries 'ranking_low_ctr': True
'stale_but_visible' or 'thin_but_visible' anywhere in top 20: True



,action_rank,content_id,reason_codes,suggested_action,is_declining_label,impressions_90d,avg_position,ctr,days_since_last_update,word_count
0,1,content_cf56e2e2e282,stale_but_visible|rankable_position|ranking_lo...,refresh_title_and_meta,1,61678.0,19.7,0.15,194.0,5125.0
1,2,content_0a91db491d14,stale_but_visible|rankable_position|ranking_lo...,refresh_title_and_meta,1,13299.0,10.5,0.49,193.0,3478.0
2,3,content_c2d929d83eaa,stale_but_visible|rankable_position|ranking_lo...,refresh_title_and_meta,1,7558.0,17.9,0.20,193.0,4758.0
3,4,content_fe16a55cd13d,stale_but_visible|rankable_position|ranking_lo...,refresh_title_and_meta,1,4556.0,16.4,0.33,194.0,3388.0
4,5,content_fe16a55cd13d,stale_but_visible|rankable_position|ranking_lo...,refresh_title_and_meta,1,4556.0,16.4,0.33,194.0,3388.0
5,6,content_5fe46e04994d,rankable_position|ranking_low_ctr,refresh_title_and_meta,1,517715.0,4.2,0.14,104.0,NaN
6,7,content_5fe46e04994d,rankable_position|ranking_low_ctr,refresh_title_and_meta,1,517715.0,4.2,0.14,104.0,NaN
7,8,content_aaef01a50def,rankable_position|ranking_low_ctr,refresh_title_and_meta,0,517109.0,5.4,0.25,22.0,NaN
8,9,content_8c19996aa890,rankable_position|ranking_low_ctr,refresh_title_and_meta,1,509252.0,2.5,0.15,20.0,2895.0
9,10,content_4c36c775b818,rankable_position|ranking_low_ctr,refresh_title_and_meta,1,463103.0,2.3,0.41,20.0,3097.0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks.** 7 of the top 20 (ranks 6, 10, 12, 13, 16, 17, 18) are not actually declining.
All seven share the same shape: very large `impressions_90d`, an already-good `avg_position`,
and a `ctr` at or near 0.0-0.4%. That's the real weakness in this baseline: my
`log1p(impressions)` term is strong enough that the top of the queue is effectively "biggest
pages with the lowest CTR," and `thin_but_visible` never gets a chance to surface at all in the
top 20 -- `stale_but_visible` only shows up riding along on pages that already qualify on the
other two reasons. Zooming out confirms this isn't just a top-20 fluke:
**Precision@50 is 0.50, actually *below* the 0.54 base rate** -- past the top ~20-30, the rule
does no better than picking pages at random. I'd fix this by capping or converting the
impressions multiplier to a percentile rank instead of raw `log1p`, so `stale_but_visible` and
`thin_but_visible` pages can surface on their own merits without also needing to be a top-scale
page.

A second, smaller pattern: several of the wrong picks (ranks 12, 13, 16) have a CTR that's
implausibly low for their position (e.g. rank 13: position 2.3 with 0.03% CTR). That reads more
like a measurement gap in the export than a genuine content problem -- worth flagging before
trusting `ranking_low_ctr` as a strong signal on its own.

**Leakage check.**

- **No product flags used.** `provider_used` and `model_used` (which LLM generated the article)
  never enter the score, the reason codes, or the suggested action -- confirmed programmatically
  below by checking the score's actual inputs.
- **No label-source columns used.** `trend_direction` and `trend_pct` -- the columns
  `is_declining_label` is literally built from -- are only read once, to build the label for
  *evaluation*, and never appear on the right-hand side of `action_score`, `reason_codes`, or
  `suggested_action`. Checked directly below by inspecting the source of those functions.
- **No future windows.** Every column the rule reads (`impressions_90d`, `days_since_last_update`,
  `word_count`, `avg_position`, `ctr`) is a trailing/point-in-time value as of the export -- there
  is no forward-looking or post-window column in this starter CSV to accidentally pull in (that
  risk lives in the warehouse release's `_prev30` vs `_last30` windows, not in this slice).
- **IDs used for grouping only.** `content_id` / `client_id` appear in the output for
  identification and appear in no arithmetic anywhere in the score.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import inspect

score_inputs = {"impressions_90d", "days_since_last_update", "word_count", "avg_position", "ctr"}
banned_as_features = {"trend_direction", "trend_pct", "provider_used", "model_used", "content_id", "client_id"}

print("Score inputs used:", sorted(score_inputs))
print("Banned columns touching the score inputs:", score_inputs & banned_as_features)
assert not (score_inputs & banned_as_features), "Leakage: a banned column is in the score inputs"

# is_declining_label is read from trend_direction ONLY for evaluation -- confirm it's never
# joined back into action_score, reason_codes or suggested_action by checking those columns
# don't reference the label or trend_direction/trend_pct anywhere in their construction.
score_source = inspect.getsource(reason_codes) + inspect.getsource(suggested_action)
for banned in ["trend_direction", "trend_pct", "is_declining_label", "provider_used", "model_used"]:
    assert banned not in score_source, f"{banned} leaked into reason_codes/suggested_action logic"
print("reason_codes / suggested_action functions contain no banned columns: confirmed")

print()
print(f"Base rate:      {base_rate:.3f}")
print(f"Precision@20:   {p20:.3f}  (lift over base rate: {p20 - base_rate:+.3f})")
print(f"Precision@50:   {p50:.3f}  (lift over base rate: {p50 - base_rate:+.3f})  <- weaker than the base rate")
print()
print("Wrong picks in top 20 (label says NOT declining):")
print(wrong_picks[["action_rank", "content_id", "impressions_90d", "avg_position", "ctr"]].to_string(index=False))


Score inputs used: ['avg_position', 'ctr', 'days_since_last_update', 'impressions_90d', 'word_count']
Banned columns touching the score inputs: set()
reason_codes / suggested_action functions contain no banned columns: confirmed

Base rate:      0.541
Precision@20:   0.650  (lift over base rate: +0.109)
Precision@50:   0.500  (lift over base rate: -0.041)  <- weaker than the base rate

Wrong picks in top 20 (label says NOT declining):
 action_rank           content_id  impressions_90d  avg_position  ctr
           8 content_aaef01a50def         517109.0           5.4 0.25
          12 content_db5989a78dd3         345111.0           5.4 0.21
          14 content_36ff89c8214e         295097.0           7.3 0.05
          15 content_8451fc6f034d         272144.0           2.3 0.03
          16 content_8451fc6f034d         272144.0           2.3 0.03
          19 content_c84a0ab98e90         223271.0           7.8 0.03
          20 content_c84a0ab98e90         223271.0           7.8 0.03


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.